# EVE Optimizer — Comprehensive Benchmark

**EVE: Evolutionary Virtual Exploration** — adaptive optimization via offspring
direction selection with probe-based fitness evaluation.

This notebook benchmarks EVE (`K=1` ≡ AdamW, `K=4` default) against commonly
adopted optimizers across multiple datasets, model architectures, and metrics.

### Optimizers compared
| Optimizer | Key idea |
|-----------|----------|
| **SGD + Momentum** | Baseline first-order method |
| **Adam** | Adaptive per-param learning rates |
| **AdamW** | Adam with decoupled weight decay |
| **EVE K=1** | Exactly AdamW (sanity check) |
| **EVE K=4** | Full evolutionary selection |

### Datasets
- **CIFAR-10** (10 classes, 50 k train / 10 k test)
- **CIFAR-100** (100 classes, 50 k train / 10 k test)
- **FashionMNIST** (10 classes, 60 k train / 10 k test)

### Model architectures
- **MLP** — 3-layer fully-connected
- **CNN** — custom 4-block ConvNet
- **ResNet-18** — standard torchvision ResNet adapted for 32×32
- **Vision Transformer (ViT-Tiny)** — patch-based transformer encoder
- **LSTM** — sequence model on flattened image rows

### Metrics
- Training loss curve
- Validation accuracy curve
- Wall-clock time per epoch
- Time to target accuracy (convergence speed)
- Final test accuracy

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────
# Clone the repo and install EVE (run this cell once on Colab).
!git clone https://github.com/SattamAltwaim/EVE.git /content/EVE 2>/dev/null || true
import sys, os
if "/content/EVE" not in sys.path:
    sys.path.insert(0, "/content/EVE")

# Quick sanity import
from eve_optimizer import EVE
print(f"EVE imported successfully (version {EVE.__module__})")

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────
import copy
import math
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T

from eve_optimizer import EVE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  Mem:  {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── 2. Configuration ──────────────────────────────────────────────────────

@dataclass
class BenchConfig:
    epochs: int = 50
    batch_size: int = 128
    num_workers: int = 2
    seed: int = 42
    log_every: int = 50        # print every N batches
    # Per-optimizer learning rates (tuned defaults)
    lr_map: Dict[str, float] = field(default_factory=lambda: {
        "SGD":      0.1,
        "Adam":     1e-3,
        "AdamW":    1e-3,
        "EVE_K1":   1e-3,
        "EVE_K4":   1e-3,
    })
    weight_decay: float = 0.01
    sgd_momentum: float = 0.9

CFG = BenchConfig()
print("Config:", CFG)

## 3. Datasets

In [ ]:
# ── 3. Dataset loading ────────────────────────────────────────────────────

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

def get_dataloaders(dataset_name: str, batch_size: int = 128, num_workers: int = 2):
    """Return (train_loader, test_loader, num_classes, input_shape)."""

    if dataset_name == "CIFAR-10":
        train_tf = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ])
        test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
        train_ds = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
        test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
        num_classes, input_shape = 10, (3, 32, 32)

    elif dataset_name == "CIFAR-100":
        train_tf = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ])
        test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
        train_ds = torchvision.datasets.CIFAR100("./data", train=True,  download=True, transform=train_tf)
        test_ds  = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=test_tf)
        num_classes, input_shape = 100, (3, 32, 32)

    elif dataset_name == "FashionMNIST":
        train_tf = T.Compose([
            T.Pad(2),  # 28→32
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,)),
        ])
        test_tf = T.Compose([T.Pad(2), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
        train_ds = torchvision.datasets.FashionMNIST("./data", train=True,  download=True, transform=train_tf)
        test_ds  = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=test_tf)
        num_classes, input_shape = 10, (1, 32, 32)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size * 2, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader, num_classes, input_shape

# Quick smoke-test
_ = get_dataloaders("CIFAR-10", batch_size=4, num_workers=0)
print("Datasets OK")

## 4. Model Architectures

Five architectures covering FC, convolutional, recurrent, and attention-based models.

In [ ]:
# ── 4. Model definitions ──────────────────────────────────────────────────

# ---------- 4a. MLP ----------
class MLP(nn.Module):
    def __init__(self, in_dim: int, num_classes: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)


# ---------- 4b. CNN (custom 4-block ConvNet) ----------
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, pool=False):
        super().__init__()
        layers = [
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class CNN(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 64),
            ConvBlock(64, 128, pool=True),
            ConvBlock(128, 256, pool=True),
            ConvBlock(256, 256, pool=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


# ---------- 4c. ResNet-18 (adapted for 32×32) ----------
def make_resnet18(in_channels: int, num_classes: int) -> nn.Module:
    model = torchvision.models.resnet18(weights=None, num_classes=num_classes)
    if in_channels != 3:
        model.conv1 = nn.Conv2d(in_channels, 64, 3, stride=1, padding=1, bias=False)
    else:
        model.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


# ---------- 4d. Vision Transformer (ViT-Tiny) ----------
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels: int, patch_size: int, d_model: int, img_size: int = 32):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, d_model, patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model) * 0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x).flatten(2).transpose(1, 2)        # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)                     # (B, N+1, D)
        return x + self.pos_embed

class ViTTiny(nn.Module):
    def __init__(self, in_channels: int, num_classes: int,
                 patch_size: int = 4, d_model: int = 128, nhead: int = 4,
                 num_layers: int = 4, mlp_ratio: int = 2):
        super().__init__()
        self.embed = PatchEmbedding(in_channels, patch_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * mlp_ratio,
            dropout=0.1, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        x = self.norm(x[:, 0])
        return self.head(x)


# ---------- 4e. LSTM (image rows as sequence) ----------
class ImageLSTM(nn.Module):
    """Treats each row of the image as a time step."""
    def __init__(self, in_channels: int, img_size: int, num_classes: int,
                 hidden_size: int = 128, num_layers: int = 2):
        super().__init__()
        self.input_size = in_channels * img_size
        self.lstm = nn.LSTM(self.input_size, hidden_size,
                            num_layers=num_layers, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 1, 3).reshape(B, H, C * W)  # (B, H, C*W)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])


# ---------- Factory ----------
def make_model(arch: str, in_channels: int, num_classes: int, img_size: int = 32) -> nn.Module:
    if arch == "MLP":
        return MLP(in_channels * img_size * img_size, num_classes)
    elif arch == "CNN":
        return CNN(in_channels, num_classes)
    elif arch == "ResNet-18":
        return make_resnet18(in_channels, num_classes)
    elif arch == "ViT-Tiny":
        return ViTTiny(in_channels, num_classes)
    elif arch == "LSTM":
        return ImageLSTM(in_channels, img_size, num_classes)
    else:
        raise ValueError(f"Unknown architecture: {arch}")

# Quick test
_m = make_model("ViT-Tiny", 3, 10)
_x = torch.randn(2, 3, 32, 32)
print("ViT-Tiny out:", _m(_x).shape)
del _m, _x
print("All model factories OK")

## 5. Training Infrastructure

In [ ]:
# ── 5. Optimizer factory + training loop ──────────────────────────────────

def make_optimizer(name: str, model: nn.Module, cfg: BenchConfig) -> torch.optim.Optimizer:
    lr = cfg.lr_map[name]
    wd = cfg.weight_decay
    if name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=cfg.sgd_momentum, weight_decay=wd)
    elif name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif name == "AdamW":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif name == "EVE_K1":
        return EVE(model.parameters(), lr=lr, weight_decay=wd, K=1)
    elif name == "EVE_K4":
        return EVE(model.parameters(), lr=lr, weight_decay=wd, K=4)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


@dataclass
class EpochMetrics:
    train_loss: float
    train_acc: float
    val_loss: float
    val_acc: float
    epoch_time: float            # wall-clock seconds
    epoch: int


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[float, float]:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    opt_name: str,
    epoch: int,
    log_every: int = 50,
) -> Tuple[float, float]:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    is_eve_k_gt1 = opt_name == "EVE_K4"

    for batch_idx, (x, y) in enumerate(loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()

        if is_eve_k_gt1:
            optimizer.step(
                model=model,
                loss_fn=lambda out, tgt: F.cross_entropy(out, tgt),
                data=(x, y),
            )
        else:
            optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def run_training(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    opt_name: str,
    epochs: int,
    device: torch.device,
) -> List[EpochMetrics]:
    history: List[EpochMetrics] = []
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device, opt_name, ep)
        val_loss, val_acc = evaluate(model, test_loader, device)
        elapsed = time.perf_counter() - t0

        history.append(EpochMetrics(
            train_loss=tr_loss, train_acc=tr_acc,
            val_loss=val_loss, val_acc=val_acc,
            epoch_time=elapsed, epoch=ep,
        ))
        if ep % 10 == 0 or ep == 1:
            print(f"  [Epoch {ep:3d}] train_loss={tr_loss:.4f} "
                  f"val_acc={val_acc:.4f}  ({elapsed:.1f}s)")
    return history

print("Training infrastructure ready")

## 6. Benchmark Runner

Runs every `(dataset × architecture × optimizer)` combination, recording full
training histories for later analysis.

In [ ]:
# ── 6. Benchmark runner ───────────────────────────────────────────────────

DATASETS = ["CIFAR-10", "CIFAR-100", "FashionMNIST"]
ARCHS    = ["MLP", "CNN", "ResNet-18", "ViT-Tiny", "LSTM"]
OPTS     = ["SGD", "Adam", "AdamW", "EVE_K1", "EVE_K4"]

# results[dataset][arch][opt_name] = List[EpochMetrics]
results: Dict[str, Dict[str, Dict[str, List[EpochMetrics]]]] = defaultdict(
    lambda: defaultdict(dict)
)

def run_benchmark(
    datasets: List[str] = DATASETS,
    archs: List[str] = ARCHS,
    opts: List[str] = OPTS,
    cfg: BenchConfig = CFG,
):
    for ds_name in datasets:
        print(f"\n{'='*70}")
        print(f"  DATASET: {ds_name}")
        print(f"{'='*70}")
        train_loader, test_loader, num_classes, (C, H, W) = get_dataloaders(
            ds_name, cfg.batch_size, cfg.num_workers
        )

        for arch in archs:
            print(f"\n  ── Architecture: {arch} ──")
            for opt_name in opts:
                print(f"\n    Optimizer: {opt_name}")
                torch.manual_seed(cfg.seed)
                torch.cuda.manual_seed_all(cfg.seed)

                model = make_model(arch, C, num_classes, H).to(DEVICE)
                optimizer = make_optimizer(opt_name, model, cfg)

                history = run_training(
                    model, train_loader, test_loader, optimizer,
                    opt_name, cfg.epochs, DEVICE,
                )
                results[ds_name][arch][opt_name] = history

                best_val = max(h.val_acc for h in history)
                total_time = sum(h.epoch_time for h in history)
                print(f"    → best val_acc={best_val:.4f}  "
                      f"total_time={total_time:.1f}s")

                del model, optimizer
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()

    print("\n✓ All benchmarks complete.")
    return results

## 7. Run the Benchmarks

**Estimated runtime on A100**: ~30–60 minutes for all 75 combinations
(3 datasets × 5 architectures × 5 optimizers × 50 epochs).

In [ ]:
# ── 7. Execute ────────────────────────────────────────────────────────────
results = run_benchmark()

## 8. Results — Summary Tables

In [ ]:
# ── 8. Summary tables ─────────────────────────────────────────────────────

def time_to_target(history: List[EpochMetrics], target_acc: float) -> Optional[float]:
    """Return cumulative wall-clock time (seconds) to first reach target_acc."""
    cum = 0.0
    for h in history:
        cum += h.epoch_time
        if h.val_acc >= target_acc:
            return cum
    return None  # never reached

def print_summary_table(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    header = f"{'Dataset':<14} {'Arch':<12} " + "".join(f"{o:<14}" for o in opts)
    print("=" * len(header))
    print("  BEST VALIDATION ACCURACY")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for ds in datasets:
        for arch in archs:
            row = f"{ds:<14} {arch:<12} "
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if hist:
                    best = max(h.val_acc for h in hist)
                    row += f"{best:.4f}       "
                else:
                    row += f"{'N/A':<14}"
            print(row)
    print()

    print("=" * len(header))
    print("  TOTAL TRAINING TIME (seconds)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for ds in datasets:
        for arch in archs:
            row = f"{ds:<14} {arch:<12} "
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if hist:
                    t = sum(h.epoch_time for h in hist)
                    row += f"{t:>7.1f}       "
                else:
                    row += f"{'N/A':<14}"
            print(row)
    print()

    print("=" * len(header))
    print("  AVERAGE TIME PER EPOCH (seconds)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for ds in datasets:
        for arch in archs:
            row = f"{ds:<14} {arch:<12} "
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if hist:
                    t = sum(h.epoch_time for h in hist) / len(hist)
                    row += f"{t:>7.2f}       "
                else:
                    row += f"{'N/A':<14}"
            print(row)
    print()

print_summary_table(results)

## 9. Visualization — Training Curves

### 9a. Validation Accuracy Curves (per dataset, one subplot per architecture)

In [ ]:
# ── 9a. Validation accuracy curves ────────────────────────────────────────

OPT_COLORS = {
    "SGD":    "#1f77b4",
    "Adam":   "#ff7f0e",
    "AdamW":  "#2ca02c",
    "EVE_K1": "#9467bd",
    "EVE_K4": "#d62728",
}
OPT_STYLES = {
    "SGD":    "--",
    "Adam":   "-.",
    "AdamW":  ":",
    "EVE_K1": "-",
    "EVE_K4": "-",
}

def plot_val_accuracy(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    for ds in datasets:
        fig, axes = plt.subplots(1, len(archs), figsize=(5 * len(archs), 4), sharey=False)
        if len(archs) == 1:
            axes = [axes]
        fig.suptitle(f"{ds} — Validation Accuracy", fontsize=14, fontweight="bold", y=1.02)

        for ax, arch in zip(axes, archs):
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                epochs = [h.epoch for h in hist]
                accs   = [h.val_acc for h in hist]
                ax.plot(epochs, accs,
                        label=opt, color=OPT_COLORS[opt],
                        linestyle=OPT_STYLES[opt],
                        linewidth=2.5 if "EVE_K4" in opt else 1.5)
            ax.set_title(arch, fontsize=11)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Val Accuracy")
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7, loc="lower right")

        fig.tight_layout()
        plt.show()

plot_val_accuracy(results)

### 9b. Training Loss Curves

In [ ]:
# ── 9b. Training loss curves ──────────────────────────────────────────────

def plot_train_loss(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    for ds in datasets:
        fig, axes = plt.subplots(1, len(archs), figsize=(5 * len(archs), 4), sharey=False)
        if len(archs) == 1:
            axes = [axes]
        fig.suptitle(f"{ds} — Training Loss", fontsize=14, fontweight="bold", y=1.02)

        for ax, arch in zip(axes, archs):
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                epochs = [h.epoch for h in hist]
                losses = [h.train_loss for h in hist]
                ax.plot(epochs, losses,
                        label=opt, color=OPT_COLORS[opt],
                        linestyle=OPT_STYLES[opt],
                        linewidth=2.5 if "EVE_K4" in opt else 1.5)
            ax.set_title(arch, fontsize=11)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Train Loss")
            ax.set_yscale("log")
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7, loc="upper right")

        fig.tight_layout()
        plt.show()

plot_train_loss(results)

### 9c. Time Per Epoch (bar chart)

In [ ]:
# ── 9c. Average time per epoch (bar chart) ────────────────────────────────

def plot_time_per_epoch(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    for ds in datasets:
        fig, axes = plt.subplots(1, len(archs), figsize=(5 * len(archs), 4), sharey=False)
        if len(archs) == 1:
            axes = [axes]
        fig.suptitle(f"{ds} — Avg Time per Epoch", fontsize=14, fontweight="bold", y=1.02)

        for ax, arch in zip(axes, archs):
            times = []
            labels = []
            colors = []
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                avg_t = sum(h.epoch_time for h in hist) / len(hist)
                times.append(avg_t)
                labels.append(opt)
                colors.append(OPT_COLORS[opt])
            ax.bar(labels, times, color=colors, edgecolor="black", linewidth=0.5)
            ax.set_title(arch, fontsize=11)
            ax.set_ylabel("Seconds / Epoch")
            ax.tick_params(axis="x", rotation=30)
            ax.grid(axis="y", alpha=0.3)

        fig.tight_layout()
        plt.show()

plot_time_per_epoch(results)

### 9d. Time to Target Accuracy (convergence speed)

For each dataset we pick a reasonable target accuracy and measure how many
wall-clock seconds each optimizer needs to first reach it.

In [ ]:
# ── 9d. Time to target accuracy ───────────────────────────────────────────

TARGET_ACC = {
    "CIFAR-10":     0.85,
    "CIFAR-100":    0.55,
    "FashionMNIST": 0.88,
}

def plot_time_to_target(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    for ds in datasets:
        target = TARGET_ACC[ds]
        fig, axes = plt.subplots(1, len(archs), figsize=(5 * len(archs), 4), sharey=False)
        if len(archs) == 1:
            axes = [axes]
        fig.suptitle(f"{ds} — Time to {target*100:.0f}% Val Accuracy",
                     fontsize=14, fontweight="bold", y=1.02)

        for ax, arch in zip(axes, archs):
            times = []
            labels = []
            colors = []
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                t = time_to_target(hist, target)
                times.append(t if t is not None else float("nan"))
                labels.append(opt)
                colors.append(OPT_COLORS[opt])

            valid = [t for t in times if not math.isnan(t)]
            if not valid:
                ax.set_title(f"{arch}\n(none reached target)", fontsize=10)
                continue

            bars = ax.bar(labels, times, color=colors, edgecolor="black", linewidth=0.5)
            for bar, t in zip(bars, times):
                if math.isnan(t):
                    bar.set_height(0)
                    bar.set_edgecolor("red")
                    ax.annotate("N/A", xy=(bar.get_x() + bar.get_width()/2, 0),
                                ha="center", va="bottom", fontsize=8, color="red")
            ax.set_title(arch, fontsize=11)
            ax.set_ylabel("Seconds to Target")
            ax.tick_params(axis="x", rotation=30)
            ax.grid(axis="y", alpha=0.3)

        fig.tight_layout()
        plt.show()

plot_time_to_target(results)

### 9e. Final Accuracy Heatmap (all combinations)

In [ ]:
# ── 9e. Final accuracy heatmap ────────────────────────────────────────────

def plot_accuracy_heatmap(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    for ds in datasets:
        data = np.full((len(archs), len(opts)), np.nan)
        for i, arch in enumerate(archs):
            for j, opt in enumerate(opts):
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if hist:
                    data[i, j] = max(h.val_acc for h in hist)

        fig, ax = plt.subplots(figsize=(8, 4))
        im = ax.imshow(data, cmap="RdYlGn", vmin=data[np.isfinite(data)].min() - 0.02,
                        vmax=data[np.isfinite(data)].max() + 0.02, aspect="auto")
        ax.set_xticks(range(len(opts)))
        ax.set_xticklabels(opts, rotation=30, ha="right")
        ax.set_yticks(range(len(archs)))
        ax.set_yticklabels(archs)
        ax.set_title(f"{ds} — Best Validation Accuracy", fontweight="bold")

        for i in range(len(archs)):
            for j in range(len(opts)):
                val = data[i, j]
                if np.isfinite(val):
                    ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                            fontsize=9, fontweight="bold",
                            color="white" if val < (data[np.isfinite(data)].mean()) else "black")

        fig.colorbar(im, ax=ax, shrink=0.8, label="Accuracy")
        fig.tight_layout()
        plt.show()

plot_accuracy_heatmap(results)

### 9f. EVE K=1 vs AdamW Sanity Check

Proposition 2 from the paper: EVE at K=1 is *exactly* AdamW.
Both curves should be identical.

In [ ]:
# ── 9f. EVE K=1 vs AdamW overlay ──────────────────────────────────────────

def plot_k1_vs_adamw(results, datasets=DATASETS, archs=ARCHS):
    for ds in datasets:
        fig, axes = plt.subplots(1, len(archs), figsize=(5 * len(archs), 4), sharey=False)
        if len(archs) == 1:
            axes = [axes]
        fig.suptitle(f"{ds} — EVE K=1 vs AdamW (should overlap)",
                     fontsize=14, fontweight="bold", y=1.02)

        for ax, arch in zip(axes, archs):
            for opt, ls, lw, color in [
                ("AdamW",  "-",  2.5, OPT_COLORS["AdamW"]),
                ("EVE_K1", "--", 1.8, OPT_COLORS["EVE_K1"]),
            ]:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                ax.plot([h.epoch for h in hist], [h.val_acc for h in hist],
                        label=opt, color=color, linestyle=ls, linewidth=lw)
            ax.set_title(arch, fontsize=11)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Val Accuracy")
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

plot_k1_vs_adamw(results)

### 9g. Overhead Analysis

Wall-clock overhead of EVE K=4 vs AdamW per architecture, expressed as
a percentage.  The paper predicts ~33% overhead.

In [ ]:
# ── 9g. Overhead analysis ─────────────────────────────────────────────────

def plot_overhead(results, datasets=DATASETS, archs=ARCHS):
    rows = []
    for ds in datasets:
        for arch in archs:
            h_adamw = results.get(ds, {}).get(arch, {}).get("AdamW")
            h_eve4  = results.get(ds, {}).get(arch, {}).get("EVE_K4")
            if h_adamw and h_eve4:
                t_adamw = sum(h.epoch_time for h in h_adamw) / len(h_adamw)
                t_eve4  = sum(h.epoch_time for h in h_eve4)  / len(h_eve4)
                overhead = (t_eve4 / t_adamw - 1) * 100
                rows.append((ds, arch, t_adamw, t_eve4, overhead))

    fig, ax = plt.subplots(figsize=(10, 5))
    labels = [f"{r[0]}\n{r[1]}" for r in rows]
    overheads = [r[4] for r in rows]
    colors = ["#d62728" if o > 50 else "#ff7f0e" if o > 33 else "#2ca02c" for o in overheads]
    bars = ax.bar(range(len(rows)), overheads, color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(33, color="gray", linestyle="--", linewidth=1, label="Paper prediction (33%)")
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
    ax.set_ylabel("Overhead %  (EVE K=4 vs AdamW)")
    ax.set_title("EVE K=4 Wall-Clock Overhead", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    for bar, pct in zip(bars, overheads):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{pct:.0f}%", ha="center", va="bottom", fontsize=8)

    fig.tight_layout()
    plt.show()

    print("\nOverhead table:")
    print(f"{'Dataset':<14} {'Arch':<12} {'AdamW (s/ep)':>13} {'EVE K4 (s/ep)':>14} {'Overhead':>10}")
    print("-" * 66)
    for ds, arch, ta, te, o in rows:
        print(f"{ds:<14} {arch:<12} {ta:>13.2f} {te:>14.2f} {o:>9.1f}%")

plot_overhead(results)

### 9h. Convergence Speed: Epochs to Target (complement to wall-clock)

In [ ]:
# ── 9h. Epochs to target accuracy ─────────────────────────────────────────

def epochs_to_target(history: List[EpochMetrics], target: float) -> Optional[int]:
    for h in history:
        if h.val_acc >= target:
            return h.epoch
    return None

def plot_epochs_to_target(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    for ds in datasets:
        target = TARGET_ACC[ds]
        fig, axes = plt.subplots(1, len(archs), figsize=(5 * len(archs), 4), sharey=False)
        if len(archs) == 1:
            axes = [axes]
        fig.suptitle(f"{ds} — Epochs to {target*100:.0f}% Val Accuracy",
                     fontsize=14, fontweight="bold", y=1.02)

        for ax, arch in zip(axes, archs):
            ep_vals = []
            labels = []
            colors = []
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                ep = epochs_to_target(hist, target)
                ep_vals.append(ep if ep is not None else float("nan"))
                labels.append(opt)
                colors.append(OPT_COLORS[opt])

            valid = [e for e in ep_vals if not math.isnan(e)]
            if not valid:
                ax.set_title(f"{arch}\n(none reached target)", fontsize=10)
                continue

            bars = ax.bar(labels, ep_vals, color=colors, edgecolor="black", linewidth=0.5)
            for bar, ep in zip(bars, ep_vals):
                if math.isnan(ep):
                    bar.set_height(0)
                    ax.annotate("N/A", xy=(bar.get_x() + bar.get_width()/2, 0),
                                ha="center", va="bottom", fontsize=8, color="red")
            ax.set_title(arch, fontsize=11)
            ax.set_ylabel("Epochs to Target")
            ax.tick_params(axis="x", rotation=30)
            ax.grid(axis="y", alpha=0.3)

        fig.tight_layout()
        plt.show()

plot_epochs_to_target(results)

## 10. Grand Summary

Final comparison table aggregated across all experiments.

In [ ]:
# ── 10. Grand summary ─────────────────────────────────────────────────────

def grand_summary(results, datasets=DATASETS, archs=ARCHS, opts=OPTS):
    print("=" * 80)
    print("  GRAND SUMMARY — Average metrics across all (dataset × architecture) combos")
    print("=" * 80)

    opt_best_accs  = defaultdict(list)
    opt_final_loss = defaultdict(list)
    opt_avg_time   = defaultdict(list)

    for ds in datasets:
        for arch in archs:
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if not hist:
                    continue
                opt_best_accs[opt].append(max(h.val_acc for h in hist))
                opt_final_loss[opt].append(hist[-1].train_loss)
                opt_avg_time[opt].append(
                    sum(h.epoch_time for h in hist) / len(hist)
                )

    print(f"\n{'Optimizer':<12} {'Avg Best ValAcc':>16} {'Avg Final Loss':>16} "
          f"{'Avg s/Epoch':>13} {'Wins (best acc)':>16}")
    print("-" * 78)

    # Count wins: how many (dataset, arch) combos each optimizer has the best accuracy
    win_counts = defaultdict(int)
    for ds in datasets:
        for arch in archs:
            best_opt, best_acc = None, -1
            for opt in opts:
                hist = results.get(ds, {}).get(arch, {}).get(opt)
                if hist:
                    ba = max(h.val_acc for h in hist)
                    if ba > best_acc:
                        best_acc = ba
                        best_opt = opt
            if best_opt:
                win_counts[best_opt] += 1

    for opt in opts:
        avg_acc  = np.mean(opt_best_accs[opt])  if opt_best_accs[opt]  else 0
        avg_loss = np.mean(opt_final_loss[opt]) if opt_final_loss[opt] else 0
        avg_time = np.mean(opt_avg_time[opt])   if opt_avg_time[opt]   else 0
        wins     = win_counts.get(opt, 0)
        total    = len(datasets) * len(archs)
        print(f"{opt:<12} {avg_acc:>16.4f} {avg_loss:>16.4f} "
              f"{avg_time:>13.2f} {wins:>8}/{total}")

    print()

grand_summary(results)
print("Benchmark complete.")